<a href="https://colab.research.google.com/github/victoriashushpannikova/nlp_homeworks/blob/main/%D0%BA%D0%BE%D0%BF%D0%B8%D1%8F_rnn_homework_(lab5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


## 1. Установка и импорт библиотек

In [2]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 2. Парсинг текстового корпуса

**Ваш уникальный сайт:** https://thesismedia.ru/

**Обоснование выбора:** на сайте содержится большое количество текстов смежной тематики, поэтому он подходит для корпуса под RNN.


In [4]:
def scrape_corpus(urls):
    """
    Парсинг текстового корпуса с выбранного сайта.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7'
    }
    texts = []
    for page, url in enumerate(urls, start=1):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            soup = BeautifulSoup(response.text, 'html.parser')

            # НОЖ
            for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'li']):
                text = tag.get_text(strip=True)
                if text and len(text) > 10:
                    texts.append(text)


            print(f"Страница {page}: собрано {len(texts)} текстов")
            time.sleep(1)
        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")
    return texts
urls = [
    "https://thesismedia.ru/"
]

corpus = scrape_corpus(urls)

print(f"\nВсего собрано текстов: {len(corpus)}")
print("Примеры:", corpus[:5])

Страница 1: собрано 33 текстов

Всего собрано текстов: 33
Примеры: ['Антропология', 'Культурология', 'Религиоведение', 'Лингвистика', 'Science Art']


## 3. Подготовка данных для RNN

Токенизация, создание последовательностей, паддинг.

In [5]:
# Токенизация, создание последовательностей, паддинг.

# Создаём токенизатор
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# Преобразуем тексты в последовательности чисел
sequences = tokenizer.texts_to_sequences(corpus)

# Создаём входные и выходные данные для causal language modeling
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Паддинг
X = pad_sequences(X)

# One-hot encoding для y
vocab_size = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Размер входных данных X: {X.shape}")
print(f"Размер выходных данных y: {y.shape}")
print(f"Размер словаря: {vocab_size}")

Размер входных данных X: (330, 40)
Размер выходных данных y: (330, 332)
Размер словаря: 332


## 4. Создание и обучение модели RNN (LSTM)

In [6]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=X.shape[1]))
model.add(LSTM(150, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Обучение (можно увеличить эпохи при хороших результатах)
history = model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.0000e+00 - loss: 5.8088 - val_accuracy: 0.0000e+00 - val_loss: 5.8127
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.0795 - loss: 5.7895 - val_accuracy: 0.0000e+00 - val_loss: 5.8355
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0227 - loss: 5.7370 - val_accuracy: 0.0000e+00 - val_loss: 6.2896
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0114 - loss: 5.6334 - val_accuracy: 0.0152 - val_loss: 6.2398
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0114 - loss: 5.5331 - val_accuracy: 0.0152 - val_loss: 6.5831
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0152 - loss: 5.4109 - val_accuracy: 0.0000e+00 - val_loss: 7.2001
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0152 - loss: 5.2670 - val_accuracy: 0.0000e+00 - val_loss: 7.3869
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0152 - loss: 5.1039 - val_accuracy: 0.

## 5. Генерация текста

Функция генерации и вывод 2–3 предложений.

In [7]:
def generate_text(seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text


seed = "как устроены"
generated = generate_text(seed, next_words=5, max_sequence_len=X.shape[1])
print("Сгенерированный текст:", generated)

seed2 = "в современном"
generated2 = generate_text(seed2, next_words=5, max_sequence_len=X.shape[1])
print("Сгенерированный текст 2:", generated2)

Сгенерированный текст: как устроены рок рок рок театра театра
Сгенерированный текст 2: в современном рок рок рок рок рок


## 6. (Дополнительно, на 5 баллов) Расчёт перплексии

Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

In [8]:
# Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

# Оцениваем модель на валидационных данных
loss, accuracy = model.evaluate(X, y, verbose=0)
perplexity = np.exp(loss)

print(f"Потери (loss): {loss:.4f}")
print(f"Перплексия: {perplexity:.4f}")
print("Пояснение: перплексия показывает, насколько модель 'удивлена' тестовыми данными. Чем ниже — тем лучше.")

Потери (loss): 5.4515
Перплексия: 233.1109
Пояснение: перплексия показывает, насколько модель 'удивлена' тестовыми данными. Чем ниже — тем лучше.


## 7. GPU (не оценивается)

Убедитесь, что обучение запускалось на GPU:
```python
print("GPU доступна:", tf.config.list_physical_devices('GPU'))
```

В Google Colab: Среда выполнения - Изменить тип среды выполнения - Выберите T4 GPU.

Документация: https://www.tensorflow.org/guide/gpu

In [9]:
print("Устройства GPU:", tf.config.list_physical_devices('GPU'))

Устройства GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
